In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

Saving rbi_cbdc_sentiment_2022_10_to_2025_03.csv.xlsx to rbi_cbdc_sentiment_2022_10_to_2025_03.csv.xlsx
User uploaded file "rbi_cbdc_sentiment_2022_10_to_2025_03.csv.xlsx" with length 10885 bytes


In [ ]:
# ==========================================================
# RBI NEWS MARKET IMPACT
# ==========================================================
import os
import time
import numpy as np
import pandas as pd
import yfinance as yf
from tqdm import tqdm
import statsmodels.formula.api as smf

# ==========================================================
# USER SETTINGS
# ==========================================================
START = "2022-10-01"
END   = "2024-12-31"
PAUSE = 0.6

FACTORS_CSV   = "/content/Indian_Fama_French_Momentum_Factors_MONTHLY.csv"
SENTIMENT_XLSX = "/content/rbi_cbdc_sentiment_2022_10_to_2025_03.csv.xlsx"

PANEL_OUTPUT = "panel_cbdc_final.csv"
REG_OUTPUT   = "cbdc_interaction_only_summary.csv"

# ==========================================================
# FIRM LIST (FINAL)
# ==========================================================
firms_info = [
    ("Paytm", "PAYTM.NS"),


    ("State Bank of India", "SBIN.NS"),
    ("Punjab National Bank", "PNB.NS"),
    ("Bank of Baroda", "BANKBARODA.NS"),
    ("Union Bank of India", "UNIONBANK.NS"),
    ("Canara Bank", "CANBK.NS"),
    ("Indian Bank", "INDIANB.NS"),
    ("Bank of India", "BANKINDIA.NS"),
    ("Central Bank of India", "CENTRALBK.NS"),

    ("HDFC Bank", "HDFCBANK.NS"),
    ("ICICI Bank", "ICICIBANK.NS"),
    ("Axis Bank", "AXISBANK.NS"),
    ("Kotak Mahindra Bank", "KOTAKBANK.NS"),
    ("Yes Bank", "YESBANK.NS"),
    ("IndusInd Bank", "INDUSINDBK.NS"),
    ("Federal Bank", "FEDERALBNK.NS"),
    ("IDFC First Bank", "IDFCFIRSTB.NS"),
    ("Bandhan Bank", "BANDHANBNK.NS"),

    ("Infosys", "INFY.NS"),
    ("TCS", "TCS.NS"),
    ("Wipro", "WIPRO.NS"),
    ("Tech Mahindra", "TECHM.NS"),
    ("HCL Technologies", "HCLTECH.NS"),

]

firms = pd.DataFrame(firms_info, columns=["firm_name", "ticker"])

# ==========================================================
# HELPER FUNCTIONS
# ==========================================================
def pick_close(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join(map(str, c)) for c in df.columns]
    for c in df.columns:
        if "close" in c.lower():
            return c
    return None

def monthly_asof(df):
    df.index = pd.to_datetime(df.index)
    idx = pd.date_range(START, END, freq="M")
    rows = []
    for me in idx:
        sub = df.loc[:me]
        if sub.empty:
            rows.append((me, np.nan, 0))
        else:
            rows.append((me, sub.iloc[-1, 0], sub.shape[0]))
    out = pd.DataFrame(rows, columns=["month_end", "price", "n_trading_days"])
    out["month"] = out["month_end"].dt.to_period("M").dt.to_timestamp()
    return out[["month", "price", "n_trading_days"]]

def download_prices(ticker):
    df = yf.download(ticker, start=START, end=END, auto_adjust=True, progress=False)
    time.sleep(PAUSE)
    if df is None or df.empty:
        return None
    col = pick_close(df)
    return monthly_asof(df[[col]])

# ==========================================================
# BUILD PRICE PANEL
# ==========================================================
panels = []
for _, r in tqdm(firms.iterrows(), total=len(firms), desc="Downloading prices"):
    out = download_prices(r["ticker"])
    if out is None:
        continue
    out["firm_name"] = r["firm_name"]
    out["firm_id"] = r["ticker"]
    panels.append(out)

panel = pd.concat(panels, ignore_index=True)
panel = panel.sort_values(["firm_id", "month"])
panel["ret"] = panel.groupby("firm_id")["price"].transform(lambda x: np.log(x / x.shift(1)))

# ==========================================================
# LOAD & STANDARDIZE FAMA–FRENCH FACTORS (ROBUST)
# ==========================================================
fac = pd.read_csv(FACTORS_CSV)
fac.columns = [c.strip().upper() for c in fac.columns]

date_col = [c for c in fac.columns if "DATE" in c or "MONTH" in c][0]
fac = fac.rename(columns={date_col: "DATE"})
fac["DATE"] = pd.to_datetime(fac["DATE"], dayfirst=True)

fac = fac.set_index("DATE").resample("M").last().reset_index()
fac["month"] = fac["DATE"].dt.to_period("M").dt.to_timestamp()

rename_map = {}
if "MKT-RF" in fac.columns:
    rename_map["MKT-RF"] = "MKT_RF"
elif "MF" in fac.columns:
    rename_map["MF"] = "MKT_RF"
elif "MKT" in fac.columns:
    rename_map["MKT"] = "MKT_RF"

if "WML" in fac.columns:
    rename_map["WML"] = "MOM"

fac = fac.rename(columns=rename_map)

for c in ["MKT_RF", "SMB", "HML", "MOM"]:
    if c not in fac.columns:
        fac[c] = np.nan

fac = fac[["month", "MKT_RF", "SMB", "HML", "MOM"]]

# ==========================================================
# LOAD RBI CBDC SENTIMENT (EXCEL)
# ==========================================================
sent = pd.read_excel(SENTIMENT_XLSX)
sent.columns = [c.strip().lower() for c in sent.columns]
sent = sent.rename(columns={"sentiment": "CBDC_SENTIMENT"})
sent["month"] = pd.to_datetime(sent["month"])

# ==========================================================
# MERGE ALL
# ==========================================================
panel = (
    panel
    .merge(fac, on="month", how="left")
    .merge(sent[["month", "CBDC_SENTIMENT"]], on="month", how="left")
)

# ==========================================================
# INDUSTRY DUMMIES
# ==========================================================
panel["fintech"] = panel["firm_name"].str.contains("paytm", case=False).astype(int)
panel["public_bank"] = panel["firm_name"].str.contains("state|punjab|baroda|union|canara|indian|central", case=False).astype(int)
panel["commercial_bank"] = panel["firm_name"].str.contains("hdfc|icici|axis|kotak|yes|indusind|federal|idfc|bandhan", case=False).astype(int)
panel["it_company"] = panel["firm_name"].str.contains("infosys|tcs|wipro|tech mahindra|hcl|ltim", case=False).astype(int)

panel.to_csv(PANEL_OUTPUT, index=False)
print("✅ Panel saved:", PANEL_OUTPUT)

# ==========================================================
# INTERACTION-ONLY REGRESSIONS
# ==========================================================
panel = panel.dropna(subset=["ret", "CBDC_SENTIMENT"]).reset_index(drop=True)

formula = (
    "ret ~ MKT_RF + SMB + HML + MOM"
    " + CBDC_SENTIMENT:fintech"
    " + CBDC_SENTIMENT:public_bank"
    " + CBDC_SENTIMENT:commercial_bank"
    " + CBDC_SENTIMENT:it_company"
)

fe_formula = formula + " + C(firm_id)"

model = smf.ols(formula, panel).fit(
    cov_type="cluster", cov_kwds={"groups": panel["firm_id"]}
)

fe_model = smf.ols(fe_formula, panel).fit(
    cov_type="cluster", cov_kwds={"groups": panel["firm_id"]}
)

terms = [
    "CBDC_SENTIMENT:fintech",
    "CBDC_SENTIMENT:public_bank",
    "CBDC_SENTIMENT:commercial_bank",
    "CBDC_SENTIMENT:it_company"
]

rows = []
for t in terms:
    rows.append({
        "term": t,
        "coef": fe_model.params.get(t, np.nan),
        "std_err": fe_model.bse.get(t, np.nan),
        "p_value": fe_model.pvalues.get(t, np.nan)
    })

pd.DataFrame(rows).to_csv(REG_OUTPUT, index=False)

print("✅ Regression summary saved:", REG_OUTPUT)
print("\nALL DONE — panel + interaction regressions completed successfully.")


✅ Panel saved: panel_cbdc_final.csv
✅ Regression summary saved: cbdc_interaction_only_summary.csv

ALL DONE — panel + interaction regressions completed successfully.


In [ ]:
# ==========================================================
# PRINT COEFFICIENTS & SIGNIFICANCE (ON SCREEN)
# ==========================================================
def sig_stars(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    else:
        return ""

print("\n================ INTERACTION-ONLY RESULTS (FE MODEL) ================\n")

for term in terms:
    coef = fe_model.params.get(term, np.nan)
    pval = fe_model.pvalues.get(term, np.nan)
    se   = fe_model.bse.get(term, np.nan)
    stars = sig_stars(pval)

    print(f"{term:35s} | Coef = {coef:>8.4f} | SE = {se:>8.4f} | p-value = {pval:>7.4f} {stars}")

print("\nSignificance levels: *** p<0.01, ** p<0.05, * p<0.10")
print("=====================================================================\n")



================ INTERACTION-ONLY RESULTS (FE MODEL) ================

CBDC_SENTIMENT:fintech              | Coef =  -0.0087 | SE =   0.0043 | p-value =  0.0421 **
CBDC_SENTIMENT:public_bank          | Coef =   0.0098 | SE =   0.0099 | p-value =  0.3219 
CBDC_SENTIMENT:commercial_bank      | Coef =  -0.0069 | SE =   0.0099 | p-value =  0.4850 
CBDC_SENTIMENT:it_company           | Coef =  -0.0164 | SE =   0.0070 | p-value =  0.0196 **

Significance levels: *** p<0.01, ** p<0.05, * p<0.10



In [ ]:
# ==========================================================
# MEDIA NEWS MARKET IMPACT
# ==========================================================
import os
import time
import numpy as np
import pandas as pd
import yfinance as yf
from tqdm import tqdm
import statsmodels.formula.api as smf

# ==========================================================
# USER SETTINGS
# ==========================================================
START = "2022-10-01"
END   = "2025-03-01"
PAUSE = 0.6

FACTORS_CSV   = "/content/Indian_Fama_French_Momentum_Factors_MONTHLY.csv"
SENTIMENT_XLSX = "/content/cbdc_news_sentiment_paragraph_monthly.xlsx"

PANEL_OUTPUT = "panel_cbdc_final.csv"
REG_OUTPUT   = "cbdc_interaction_only_summary.csv"

# ==========================================================
# FIRM LIST (FINAL)
# ==========================================================
firms_info = [
    ("Paytm", "PAYTM.NS"),


    ("State Bank of India", "SBIN.NS"),
    ("Punjab National Bank", "PNB.NS"),
    ("Bank of Baroda", "BANKBARODA.NS"),
    ("Union Bank of India", "UNIONBANK.NS"),
    ("Canara Bank", "CANBK.NS"),
    ("Indian Bank", "INDIANB.NS"),
    ("Bank of India", "BANKINDIA.NS"),
    ("Central Bank of India", "CENTRALBK.NS"),

    ("HDFC Bank", "HDFCBANK.NS"),
    ("ICICI Bank", "ICICIBANK.NS"),
    ("Axis Bank", "AXISBANK.NS"),
    ("Kotak Mahindra Bank", "KOTAKBANK.NS"),
    ("Yes Bank", "YESBANK.NS"),
    ("IndusInd Bank", "INDUSINDBK.NS"),
    ("Federal Bank", "FEDERALBNK.NS"),
    ("IDFC First Bank", "IDFCFIRSTB.NS"),
    ("Bandhan Bank", "BANDHANBNK.NS"),

    ("Infosys", "INFY.NS"),
    ("TCS", "TCS.NS"),
    ("Wipro", "WIPRO.NS"),
    ("Tech Mahindra", "TECHM.NS"),
    ("HCL Technologies", "HCLTECH.NS"),

]

firms = pd.DataFrame(firms_info, columns=["firm_name", "ticker"])

# ==========================================================
# HELPER FUNCTIONS
# ==========================================================
def pick_close(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join(map(str, c)) for c in df.columns]
    for c in df.columns:
        if "close" in c.lower():
            return c
    return None

def monthly_asof(df):
    df.index = pd.to_datetime(df.index)
    idx = pd.date_range(START, END, freq="M")
    rows = []
    for me in idx:
        sub = df.loc[:me]
        if sub.empty:
            rows.append((me, np.nan, 0))
        else:
            rows.append((me, sub.iloc[-1, 0], sub.shape[0]))
    out = pd.DataFrame(rows, columns=["month_end", "price", "n_trading_days"])
    out["month"] = out["month_end"].dt.to_period("M").dt.to_timestamp()
    return out[["month", "price", "n_trading_days"]]

def download_prices(ticker):
    df = yf.download(ticker, start=START, end=END, auto_adjust=True, progress=False)
    time.sleep(PAUSE)
    if df is None or df.empty:
        return None
    col = pick_close(df)
    return monthly_asof(df[[col]])

# ==========================================================
# BUILD PRICE PANEL
# ==========================================================
panels = []
for _, r in tqdm(firms.iterrows(), total=len(firms), desc="Downloading prices"):
    out = download_prices(r["ticker"])
    if out is None:
        continue
    out["firm_name"] = r["firm_name"]
    out["firm_id"] = r["ticker"]
    panels.append(out)

panel = pd.concat(panels, ignore_index=True)
panel = panel.sort_values(["firm_id", "month"])
panel["ret"] = panel.groupby("firm_id")["price"].transform(lambda x: np.log(x / x.shift(1)))

# ==========================================================
# LOAD & STANDARDIZE FAMA–FRENCH FACTORS (ROBUST)
# ==========================================================
fac = pd.read_csv(FACTORS_CSV)
fac.columns = [c.strip().upper() for c in fac.columns]

date_col = [c for c in fac.columns if "DATE" in c or "MONTH" in c][0]
fac = fac.rename(columns={date_col: "DATE"})
fac["DATE"] = pd.to_datetime(fac["DATE"], dayfirst=True)

fac = fac.set_index("DATE").resample("M").last().reset_index()
fac["month"] = fac["DATE"].dt.to_period("M").dt.to_timestamp()

rename_map = {}
if "MKT-RF" in fac.columns:
    rename_map["MKT-RF"] = "MKT_RF"
elif "MF" in fac.columns:
    rename_map["MF"] = "MKT_RF"
elif "MKT" in fac.columns:
    rename_map["MKT"] = "MKT_RF"

if "WML" in fac.columns:
    rename_map["WML"] = "MOM"

fac = fac.rename(columns=rename_map)

for c in ["MKT_RF", "SMB", "HML", "MOM"]:
    if c not in fac.columns:
        fac[c] = np.nan

fac = fac[["month", "MKT_RF", "SMB", "HML", "MOM"]]

# ==========================================================
# LOAD RBI CBDC SENTIMENT (EXCEL)
# ==========================================================
sent = pd.read_excel(SENTIMENT_XLSX)
sent.columns = [c.strip().lower() for c in sent.columns]
sent = sent.rename(columns={"sentiment": "CBDC_SENTIMENT"})
sent["month"] = pd.to_datetime(sent["month"])

# ==========================================================
# MERGE ALL
# ==========================================================
panel = (
    panel
    .merge(fac, on="month", how="left")
    .merge(sent[["month", "CBDC_SENTIMENT"]], on="month", how="left")
)

# ==========================================================
# INDUSTRY DUMMIES
# ==========================================================
panel["fintech"] = panel["firm_name"].str.contains("paytm", case=False).astype(int)
panel["public_bank"] = panel["firm_name"].str.contains("state|punjab|baroda|union|canara|indian|central", case=False).astype(int)
panel["commercial_bank"] = panel["firm_name"].str.contains("hdfc|icici|axis|kotak|yes|indusind|federal|idfc|bandhan", case=False).astype(int)
panel["it_company"] = panel["firm_name"].str.contains("infosys|tcs|wipro|tech mahindra|hcl|ltim", case=False).astype(int)

panel.to_csv(PANEL_OUTPUT, index=False)
print("✅ Panel saved:", PANEL_OUTPUT)

# ==========================================================
# INTERACTION-ONLY REGRESSIONS
# ==========================================================
panel = panel.dropna(subset=["ret", "CBDC_SENTIMENT"]).reset_index(drop=True)

formula = (
    "ret ~ MKT_RF + SMB + HML + MOM"
    " + CBDC_SENTIMENT:fintech"
    " + CBDC_SENTIMENT:public_bank"
    " + CBDC_SENTIMENT:commercial_bank"
    " + CBDC_SENTIMENT:it_company"
)

fe_formula = formula + " + C(firm_id)"

model = smf.ols(formula, panel).fit(
    cov_type="cluster", cov_kwds={"groups": panel["firm_id"]}
)

fe_model = smf.ols(fe_formula, panel).fit(
    cov_type="cluster", cov_kwds={"groups": panel["firm_id"]}
)

terms = [
    "CBDC_SENTIMENT:fintech",
    "CBDC_SENTIMENT:public_bank",
    "CBDC_SENTIMENT:commercial_bank",
    "CBDC_SENTIMENT:it_company"
]

rows = []
for t in terms:
    rows.append({
        "term": t,
        "coef": fe_model.params.get(t, np.nan),
        "std_err": fe_model.bse.get(t, np.nan),
        "p_value": fe_model.pvalues.get(t, np.nan)
    })

pd.DataFrame(rows).to_csv(REG_OUTPUT, index=False)

print("✅ Regression summary saved:", REG_OUTPUT)
print("\nALL DONE — panel + interaction regressions completed successfully.")


✅ Panel saved: panel_cbdc_final.csv
✅ Regression summary saved: cbdc_interaction_only_summary.csv

ALL DONE — panel + interaction regressions completed successfully.


In [ ]:
# ==========================================================
# PRINT COEFFICIENTS & SIGNIFICANCE (ON SCREEN)
# ==========================================================
def sig_stars(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    else:
        return ""

print("\n================ INTERACTION-ONLY RESULTS (FE MODEL) ================\n")

for term in terms:
    coef = fe_model.params.get(term, np.nan)
    pval = fe_model.pvalues.get(term, np.nan)
    se   = fe_model.bse.get(term, np.nan)
    stars = sig_stars(pval)

    print(f"{term:35s} | Coef = {coef:>8.4f} | SE = {se:>8.4f} | p-value = {pval:>7.4f} {stars}")

print("\nSignificance levels: *** p<0.01, ** p<0.05, * p<0.10")
print("=====================================================================\n")



================ INTERACTION-ONLY RESULTS (FE MODEL) ================

CBDC_SENTIMENT:fintech              | Coef =  -0.2717 | SE =   0.0157 | p-value =  0.0000 ***
CBDC_SENTIMENT:public_bank          | Coef =   0.1017 | SE =   0.0234 | p-value =  0.0000 ***
CBDC_SENTIMENT:commercial_bank      | Coef =  -0.0412 | SE =   0.0182 | p-value =  0.0233 **
CBDC_SENTIMENT:it_company           | Coef =  -0.0298 | SE =   0.0224 | p-value =  0.1838 

Significance levels: *** p<0.01, ** p<0.05, * p<0.10

